In [1]:
from __future__ import annotations

import torch

In [2]:
BOARD_SIZE = 4

In [3]:
def board_example():
    board = torch.zeros(BOARD_SIZE, BOARD_SIZE, dtype=torch.int8)
    pos = torch.tensor([[0, 1], [3, 2], [3, 3], [2, 1]])
    board[pos[::2, 0], pos[::2, 1]] = -1
    board[pos[1::2, 0], pos[1::2, 1]] = 1
    return board

In [4]:
print(board_example())

tensor([[ 0, -1,  0,  0],
        [ 0,  0,  0,  0],
        [ 0,  1,  0,  0],
        [ 0,  0,  1, -1]], dtype=torch.int8)


In [5]:
def rotate_board(board: torch.Tensor) -> torch.Tensor:
    next_board = torch.zeros(BOARD_SIZE, BOARD_SIZE, dtype=board.dtype)
    next_board[1:3, 1:3] = board[1:3, 1:3].rot90()

    next_board[0, :3] = board[0, 1:]
    next_board[3, 1:] = board[3, :3]
    next_board[1:, 0] = board[:3, 0]
    next_board[:3, 3] = board[1:, 3]

    return next_board

In [6]:
rotate_board(board_example())

tensor([[-1,  0,  0,  0],
        [ 0,  0,  0,  0],
        [ 0,  0,  1, -1],
        [ 0,  0,  0,  1]], dtype=torch.int8)

In [7]:
from enum import IntEnum


class Player(IntEnum):
    ONE = 1
    TWO = -1

In [8]:
def has_winner(board: torch.Tensor, player: Player) -> bool:
    target_sum = player.value * BOARD_SIZE

    def hv(dim) -> bool:
        return torch.any(board.sum(dim) == target_sum).item()  # type: ignore

    def diag() -> bool:
        return (board.diag().sum() == target_sum).item() or (board.rot90().diag().sum() == target_sum).item()  # type: ignore

    return hv(dim=0) or hv(dim=1) or diag()

In [9]:
def next_player(player: Player) -> Player:
    return Player.ONE if player is Player.TWO else Player.TWO

In [10]:
from dataclasses import dataclass

Vec2I = tuple[int, int]


@dataclass(frozen=True)
class Move:
    src: Vec2I
    dst: Vec2I


@dataclass(frozen=True)
class Place:
    spot: Vec2I


class Draw:
    def __repr__(self) -> str:
        return "Draw"


@dataclass(frozen=True)
class EndOfGame:
    winner: Player | Draw


class Spin:
    def __repr__(self) -> str:
        return "Spin"


GameLog = Move | Place | EndOfGame | Spin

In [11]:
@dataclass(frozen=True)
class MinMaxStat:
    score: float
    game_logs: list[GameLog]

In [12]:
from collections.abc import Callable, Iterator
from itertools import product


class Points(IntEnum):
    WIN = 1000
    LOSE = -1000
    DRAW = 0


HashT = tuple[int, ...]
CacheT = dict[HashT, MinMaxStat]
cache: CacheT = {}


def check_winner(board: torch.Tensor, turn: Player) -> Points | None:
    current_player_win = has_winner(board, turn)
    opposite_player_win = has_winner(board, next_player(turn))
    if current_player_win and opposite_player_win:
        return Points.DRAW
    elif current_player_win:
        return Points.WIN
    elif opposite_player_win:
        return Points.LOSE
    return None


def coords() -> Iterator[Vec2I]:
    return product(range(BOARD_SIZE), range(BOARD_SIZE))  # type: ignore


def has_spots(board: torch.Tensor) -> bool:
    return bool(board.count_nonzero() != torch.tensor(BOARD_SIZE * BOARD_SIZE))


def iter_available_place(board: torch.Tensor) -> Iterator[Vec2I]:
    available = board == 0

    def is_available(pos: Vec2I) -> bool:
        return bool(available[pos])

    yield from filter(is_available, coords())


def iter_available_moves(board: torch.Tensor, player: Player) -> Iterator[Move]:
    other_player = next_player(player)

    def _player_spot(pos: Vec2I) -> bool:
        return bool(board[pos] == other_player)

    def add_pos(pos: Vec2I) -> Callable[[Vec2I], Vec2I]:
        def _curry(d: Vec2I) -> Vec2I:
            return pos[0] + d[0], pos[1] + d[1]

        return _curry

    def _free_spot(pos: Vec2I) -> bool:
        if not (0 <= pos[0] < BOARD_SIZE and 0 <= pos[1] < BOARD_SIZE):
            return False
        return bool(board[pos] == torch.tensor(0))

    def _free_adj_spots(pos: Vec2I) -> Iterator[Vec2I]:
        dirs: list[Vec2I] = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        adj_spots = map(add_pos(pos), dirs)
        yield from filter(_free_spot, adj_spots)

    player_spots = filter(_player_spot, coords())

    for src in player_spots:
        for dst in _free_adj_spots(src):
            yield Move(src, dst)


def hash_board(board: torch.Tensor, turn: Player, moved: bool) -> HashT:
    return tuple(board.flatten().tolist()) + (turn.value, 1 if moved else 0)


MORE_SPINS_AFTER_NO_SPOTS = 5


def no_spot_spins(board: torch.Tensor, player: Player, depth: int) -> MinMaxStat:
    logs: list[GameLog] = []
    for i in range(MORE_SPINS_AFTER_NO_SPOTS):
        board = rotate_board(board)
        logs.append(Spin())
        points = check_winner(board, player)
        if points is Points.WIN:
            logs.append(EndOfGame(player))
            return MinMaxStat(points.value / (depth + i + 1), logs)
        elif points is Points.LOSE:
            logs.append(EndOfGame(next_player(player)))
            return MinMaxStat(points.value / (depth + i + 1), logs)
    logs.append(EndOfGame(Draw()))
    return MinMaxStat(Points.DRAW.value, logs)


def with_log(stats: MinMaxStat, logs: list[GameLog]) -> MinMaxStat:
    return MinMaxStat(stats.score, logs + stats.game_logs)


def move_turns(board: torch.Tensor, player: Player, depth: int) -> list[MinMaxStat]:
    place_turns: list[MinMaxStat] = []
    for move in iter_available_moves(board, player):
        current = board.clone()
        current[move.src] = 0
        current[move.dst] = next_player(player)
        place_turn = minmax(current, player, depth=depth + 1, moved=True)
        place_turns.append(with_log(place_turn, [move]))
    return place_turns


def negate_scores(stats: list[MinMaxStat]) -> list[MinMaxStat]:
    return [MinMaxStat(-s.score, s.game_logs) for s in stats]

In [13]:
def rotate_pos(pos: Vec2I, k: int) -> Vec2I:
    temp_board = torch.zeros(BOARD_SIZE, BOARD_SIZE)
    temp_board[pos] = 1
    row, col = (temp_board.rot90(k) == 1).nonzero(as_tuple=True)
    return (row[0].item(), col[0].item())  # type: ignore


def rotate_log(log: GameLog, k: int) -> GameLog:
    match log:
        case Place(spot=spot):
            return Place(rotate_pos(spot, k))
        case Move(src=src, dst=dst):
            return Move(rotate_pos(src, k), rotate_pos(dst, k))
        case _:
            return log  # EndOfGame i Spin nie mają współrzędnych, zostają bez zmian


def rotate_logs(logs: list[GameLog], k: int) -> list[GameLog]:
    return [rotate_log(log, k) for log in logs]


def memo(
    cache: CacheT,
    board: torch.Tensor,
    stat: MinMaxStat,
    turn: Player,
    moved: bool,
) -> None:
    cache[hash_board(board, turn, moved)] = stat


def cache_get(
    board: torch.Tensor,
    turn: Player,
    moved: bool,
) -> MinMaxStat | None:
    for k in range(4):
        stat = cache.get(hash_board(board.rot90(k), turn, moved))
        if stat is not None:
            if k == 0:
                return stat
            return MinMaxStat(stat.score, rotate_logs(stat.game_logs, -k))
    return None

In [14]:
def minmax(
    board: torch.Tensor,
    player: Player,
    *,
    depth: int,
    moved: bool = False,
) -> MinMaxStat:
    if (stat := cache_get(board, player, moved)) is not None:
        return stat

    if not has_spots(board):
        return no_spot_spins(board, player, depth)

    next_player_turns: list[MinMaxStat] = []
    terminal_turns: list[MinMaxStat] = []
    for spot in iter_available_place(board):
        current = board.clone()
        current[spot] = player
        current = rotate_board(current)
        logs = [Place(spot), Spin()]
        points = check_winner(current, player)
        if points is Points.WIN:
            best = MinMaxStat(points.value / depth, logs + [EndOfGame(player)])
            memo(cache, board, best, player, moved)
            return best
        elif points is Points.LOSE:
            terminal_turns.append(
                MinMaxStat(
                    points.value / depth, logs + [EndOfGame(next_player(player))]
                )
            )
        elif points is Points.DRAW:
            terminal_turns.append(
                MinMaxStat(points.value / depth, logs + [EndOfGame(Draw())])
            )
        else:
            next_turn = minmax(current, next_player(player), depth=depth + 1)
            next_player_turns.append(with_log(next_turn, logs))

    best = max(
        ([] if moved else move_turns(board, player, depth))
        + terminal_turns
        + negate_scores(next_player_turns),
        key=lambda s: s.score,
    )
    memo(cache, board, best, player, moved)
    return best

In [15]:
import random


def board_example_random(each_player_turns: int = 2):
    if each_player_turns <= 0:
        return torch.zeros(4, 4)
    board = torch.zeros(4, 4, dtype=torch.int8)
    possible = list(product(range(4), range(4)))
    pos = random.sample(possible, each_player_turns * 2)
    pos = torch.tensor(pos)
    board[pos[::2, 0], pos[::2, 1]] = -1
    board[pos[1::2, 0], pos[1::2, 1]] = 1
    return board

In [16]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib.colors import ListedColormap

CMAP = ListedColormap(["#e74c3c", "#ecf0f1", "#3498db"])


def render_board(ax, board: torch.Tensor, title: str = "") -> None:
    ax.clear()
    ax.imshow(board.numpy(), cmap=CMAP, vmin=-1, vmax=1)
    ax.set_xticks(range(BOARD_SIZE))
    ax.set_yticks(range(BOARD_SIZE))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(color="white", linewidth=2)
    ax.set_title(title, fontsize=11)
    for i in range(BOARD_SIZE):
        for j in range(BOARD_SIZE):
            val = board[i, j].item()
            if val != 0:
                ax.text(j, i, "●", ha="center", va="center", fontsize=26, color="white")


def apply_log_to_board(
    board: torch.Tensor, log: GameLog, player: Player
) -> tuple[torch.Tensor, Player]:
    match log:
        case Place() as place:
            board[place.spot] = player
            player = next_player(player)
        case Move() as move:
            piece = board[move.src]
            board[move.dst] = piece
            board[move.src] = 0
        case Spin():
            board = rotate_board(board)
    return board, player


def board_history(
    board: torch.Tensor,
    game_logs: list[GameLog],
    starting_player: Player,
) -> list[torch.Tensor]:
    board = board.clone()
    boards = [board.clone()]
    player = starting_player
    for log in game_logs:
        board, player = apply_log_to_board(board, log, player)
        boards.append(board.clone())
    return boards


def make_history_viewer(
    boards_history: list[torch.Tensor],
    game_logs: list[GameLog],
) -> widgets.VBox:
    n_steps = len(boards_history)

    fig, ax = plt.subplots(figsize=(4, 4))
    plt.close(fig)
    out = widgets.Output()

    slider = widgets.IntSlider(
        min=0, max=n_steps - 1, step=1, value=0, description="Step:"
    )
    play = widgets.Play(
        min=0, max=n_steps - 1, step=1, interval=800, description="Play"
    )
    widgets.jslink((play, "value"), (slider, "value"))

    log_label = widgets.Label()

    def update(change=None):
        idx = slider.value
        log_text = str(game_logs[idx - 1]) if idx > 0 else "Starting board"
        log_label.value = f"[{idx}/{n_steps - 1}] {log_text}"
        render_board(ax, boards_history[idx], title=log_label.value)
        with out:
            out.clear_output(wait=True)
            display(fig)

    slider.observe(update, names="value")
    update()

    controls = widgets.HBox([play, slider])
    return widgets.VBox([controls, log_label, out])

In [17]:
start_board = board_example_random(each_player_turns=6)
starting_player = Player.ONE
stats = minmax(start_board, starting_player, depth=1)

viewer = make_history_viewer(
    board_history(start_board, stats.game_logs, starting_player),
    stats.game_logs,
)
display(viewer)

In [18]:
import json
from dataclasses import asdict, dataclass


@dataclass(frozen=True)
class DatasetEntry:
    board: list[list[int]]
    move: dict | None
    place: dict


def build_first_entry(
    start_board: torch.Tensor,
    game_logs: list[GameLog],
) -> DatasetEntry:
    board = start_board.clone()
    pending_move: Move | None = None

    for log in game_logs:
        match log:
            case Move() as move:
                pending_move = move
            case Place() as place:
                return DatasetEntry(
                    board=board.tolist(),
                    move=(
                        {
                            "src": list(pending_move.src),
                            "dst": list(pending_move.dst),
                        }
                        if pending_move is not None
                        else None
                    ),
                    place={"spot": list(place.spot)},
                )
            case Spin():
                board = rotate_board(board)
            case EndOfGame():
                pass

    assert False, "Should not be reachable"


def build_last_entry(
    start_board: torch.Tensor,
    game_logs: list[GameLog],
    starting_player: Player,
) -> DatasetEntry:
    board = start_board.clone()
    player = starting_player
    pending_move: Move | None = None
    last_entry: DatasetEntry | None = None

    for log in game_logs:
        match log:
            case Move() as move:
                pending_move = move
            case Place() as place:
                if player is Player.ONE:
                    last_entry = DatasetEntry(
                        board=board.tolist(),
                        move=(
                            {
                                "src": list(pending_move.src),
                                "dst": list(pending_move.dst),
                            }
                            if pending_move is not None
                            else None
                        ),
                        place={"spot": list(place.spot)},
                    )
                if pending_move is not None:
                    board[pending_move.src] = 0
                    board[pending_move.dst] = next_player(player)
                    pending_move = None
                board[place.spot] = player
                player = next_player(player)
            case Spin():
                board = rotate_board(board)
            case EndOfGame():
                pass

    assert last_entry is not None, "Should not be reachable"
    return last_entry


def save_dataset_jsonl(entries: list[DatasetEntry], path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.writelines(json.dumps(asdict(entry)) + "\n" for entry in entries)

In [19]:
train_examples = 2000
entries = []
starting_player = Player.ONE

for i in range(train_examples):
    each_player_turns = random.randint(4, 6)
    start_board = board_example_random(each_player_turns)
    stats = minmax(start_board, starting_player, depth=1)
    first_entry = build_first_entry(start_board, stats.game_logs)
    entries.append(first_entry)
    last_entry = build_last_entry(start_board, stats.game_logs, starting_player)
    if last_entry != first_entry:
        entries.append(last_entry)
    if i % 10 == 0:
        print(f"{i+1}/{train_examples}")

print(f"{train_examples}/{train_examples}")
save_dataset_jsonl(entries, "orbito_dataset_first_last_4_6_1000.jsonl")

1/2000
11/2000
21/2000
31/2000
41/2000
51/2000
61/2000
71/2000
81/2000
91/2000
101/2000
111/2000
121/2000
131/2000
141/2000
151/2000
161/2000
171/2000
181/2000
191/2000
201/2000
211/2000
221/2000
231/2000
241/2000
251/2000
261/2000
271/2000
281/2000
291/2000
301/2000
311/2000
321/2000
331/2000
341/2000
351/2000
361/2000
371/2000
381/2000
391/2000
401/2000
411/2000
421/2000
431/2000
441/2000
451/2000
461/2000
471/2000
481/2000
491/2000
501/2000
511/2000
521/2000
531/2000
541/2000
551/2000
561/2000
571/2000
581/2000
591/2000
601/2000
611/2000
621/2000
631/2000
641/2000
651/2000
661/2000
671/2000
681/2000
691/2000
701/2000
711/2000
721/2000
731/2000
741/2000
751/2000
761/2000
771/2000
781/2000
791/2000
801/2000
811/2000
821/2000
831/2000
841/2000
851/2000
861/2000
871/2000
881/2000
891/2000
901/2000
911/2000
921/2000
931/2000
941/2000
951/2000
961/2000
971/2000
981/2000
991/2000
1001/2000
1011/2000
1021/2000
1031/2000
1041/2000
1051/2000
1061/2000
1071/2000
1081/2000
1091/2000
1101/2000
1

In [20]:
train_examples = 2000
entries = []
starting_player = Player.ONE

for i in range(train_examples):
    each_player_turns = random.randint(2, 4)
    start_board = board_example_random(each_player_turns)
    stats = minmax(start_board, starting_player, depth=1)
    first_entry = build_first_entry(start_board, stats.game_logs)
    entries.append(first_entry)
    last_entry = build_last_entry(start_board, stats.game_logs, starting_player)
    if last_entry != first_entry:
        entries.append(last_entry)
    if i % 10 == 0:
        print(f"{i+1}/{train_examples}")

print(f"{train_examples}/{train_examples}")
save_dataset_jsonl(entries, "orbito_dataset_first_last_2_4_2000.jsonl")

1/2000
11/2000
21/2000
31/2000
41/2000
51/2000
61/2000
71/2000
81/2000
91/2000
101/2000
111/2000
121/2000
131/2000
141/2000
151/2000
161/2000
171/2000
181/2000
191/2000
201/2000
211/2000
221/2000
231/2000
241/2000
251/2000
261/2000
271/2000
281/2000
291/2000
301/2000
311/2000
321/2000
331/2000
341/2000
351/2000
361/2000
371/2000
381/2000
391/2000
401/2000
411/2000
421/2000
431/2000
441/2000
451/2000
461/2000
471/2000
481/2000
491/2000
501/2000
511/2000
521/2000
531/2000
541/2000
551/2000
561/2000
571/2000
581/2000
591/2000
601/2000
611/2000
621/2000
631/2000
641/2000
651/2000
661/2000
671/2000
681/2000
691/2000
701/2000
711/2000
721/2000
731/2000
741/2000
751/2000
761/2000
771/2000
781/2000
791/2000
801/2000
811/2000
821/2000
831/2000
841/2000
851/2000
861/2000
871/2000
881/2000
891/2000
901/2000
911/2000
921/2000
931/2000
941/2000
951/2000
961/2000
971/2000
981/2000
991/2000
1001/2000
1011/2000
1021/2000
1031/2000
1041/2000
1051/2000
1061/2000
1071/2000
1081/2000
1091/2000
1101/2000
1

In [21]:
train_examples = 2000
entries = []
starting_player = Player.ONE

for i in range(train_examples):
    each_player_turns = random.randint(2, 6)
    start_board = board_example_random(each_player_turns)
    stats = minmax(start_board, starting_player, depth=1)
    first_entry = build_first_entry(start_board, stats.game_logs)
    entries.append(first_entry)
    last_entry = build_last_entry(start_board, stats.game_logs, starting_player)
    if last_entry != first_entry:
        entries.append(last_entry)
    if i % 10 == 0:
        print(f"{i+1}/{train_examples}")

print(f"{train_examples}/{train_examples}")
save_dataset_jsonl(entries, "orbito_dataset_first_last_2_6_2000.jsonl")

1/2000
11/2000
21/2000
31/2000
41/2000
51/2000
61/2000
71/2000
81/2000
91/2000
101/2000
111/2000
121/2000
131/2000
141/2000
151/2000
161/2000
171/2000
181/2000
191/2000
201/2000
211/2000
221/2000
231/2000
241/2000
251/2000
261/2000
271/2000
281/2000
291/2000
301/2000
311/2000
321/2000
331/2000
341/2000
351/2000
361/2000
371/2000
381/2000
391/2000
401/2000
411/2000
421/2000
431/2000
441/2000
451/2000
461/2000
471/2000
481/2000
491/2000
501/2000
511/2000
521/2000
531/2000
541/2000
551/2000
561/2000
571/2000
581/2000
591/2000
601/2000
611/2000
621/2000
631/2000
641/2000
651/2000
661/2000
671/2000
681/2000
691/2000
701/2000
711/2000
721/2000
731/2000
741/2000
751/2000
761/2000
771/2000
781/2000
791/2000
801/2000
811/2000
821/2000
831/2000
841/2000
851/2000
861/2000
871/2000
881/2000
891/2000
901/2000
911/2000
921/2000
931/2000
941/2000
951/2000
961/2000
971/2000
981/2000
991/2000
1001/2000
1011/2000
1021/2000
1031/2000
1041/2000
1051/2000
1061/2000
1071/2000
1081/2000
1091/2000
1101/2000
1

In [22]:
train_examples = 2000
entries = []
starting_player = Player.ONE

for i in range(train_examples):
    each_player_turns = random.randint(3, 5)
    start_board = board_example_random(each_player_turns)
    stats = minmax(start_board, starting_player, depth=1)
    first_entry = build_first_entry(start_board, stats.game_logs)
    entries.append(first_entry)
    last_entry = build_last_entry(start_board, stats.game_logs, starting_player)
    if last_entry != first_entry:
        entries.append(last_entry)
    if i % 10 == 0:
        print(f"{i+1}/{train_examples}")

print(f"{train_examples}/{train_examples}")
save_dataset_jsonl(entries, "orbito_dataset_first_last_3_5_2000.jsonl")

1/2000
11/2000
21/2000
31/2000
41/2000
51/2000
61/2000
71/2000
81/2000
91/2000
101/2000
111/2000
121/2000
131/2000
141/2000
151/2000
161/2000
171/2000
181/2000
191/2000
201/2000
211/2000
221/2000
231/2000
241/2000
251/2000
261/2000
271/2000
281/2000
291/2000
301/2000
311/2000
321/2000
331/2000
341/2000
351/2000
361/2000
371/2000
381/2000
391/2000
401/2000
411/2000
421/2000
431/2000
441/2000
451/2000
461/2000
471/2000
481/2000
491/2000
501/2000
511/2000
521/2000
531/2000
541/2000
551/2000
561/2000
571/2000
581/2000
591/2000
601/2000
611/2000
621/2000
631/2000
641/2000
651/2000
661/2000
671/2000
681/2000
691/2000
701/2000
711/2000
721/2000
731/2000
741/2000
751/2000
761/2000
771/2000
781/2000
791/2000
801/2000
811/2000
821/2000
831/2000
841/2000
851/2000
861/2000
871/2000
881/2000
891/2000
901/2000
911/2000
921/2000
931/2000
941/2000
951/2000
961/2000
971/2000
981/2000
991/2000
1001/2000
1011/2000
1021/2000
1031/2000
1041/2000
1051/2000
1061/2000
1071/2000
1081/2000
1091/2000
1101/2000
1

In [24]:
train_examples = 50_000
entries = []
starting_player = Player.ONE

for i in range(train_examples):
    each_player_turns = random.randint(2, 6)
    start_board = board_example_random(each_player_turns)
    stats = minmax(start_board, starting_player, depth=1)
    first_entry = build_first_entry(start_board, stats.game_logs)
    entries.append(first_entry)
    last_entry = build_last_entry(start_board, stats.game_logs, starting_player)
    if last_entry != first_entry:
        entries.append(last_entry)
    if i % 10 == 0:
        print(f"{i+1}/{train_examples}")

print(f"{train_examples}/{train_examples}")
save_dataset_jsonl(entries, "orbito_dataset_first_last_2_6_50000.jsonl")

1/50000
11/50000
21/50000
31/50000
41/50000
51/50000
61/50000
71/50000
81/50000
91/50000
101/50000
111/50000
121/50000
131/50000
141/50000
151/50000
161/50000
171/50000
181/50000
191/50000
201/50000
211/50000
221/50000
231/50000
241/50000
251/50000
261/50000
271/50000
281/50000
291/50000
301/50000
311/50000
321/50000
331/50000
341/50000
351/50000
361/50000
371/50000
381/50000
391/50000
401/50000
411/50000
421/50000
431/50000
441/50000
451/50000
461/50000
471/50000
481/50000
491/50000
501/50000
511/50000
521/50000
531/50000
541/50000
551/50000
561/50000
571/50000
581/50000
591/50000
601/50000
611/50000
621/50000
631/50000
641/50000
651/50000
661/50000
671/50000
681/50000
691/50000
701/50000
711/50000
721/50000
731/50000
741/50000
751/50000
761/50000
771/50000
781/50000
791/50000
801/50000
811/50000
821/50000
831/50000
841/50000
851/50000
861/50000
871/50000
881/50000
891/50000
901/50000
911/50000
921/50000
931/50000
941/50000
951/50000
961/50000
971/50000
981/50000
991/50000
1001/50000


In [25]:
import pickle

with open("minmax_cache.pkl", "wb") as f:
    pickle.dump(cache, f)

In [1]:
import pickle

with open("minmax_cache.pkl", "rb") as f:
    cache = pickle.load(f)

AttributeError: Can't get attribute 'MinMaxStat' on <module '__main__'>